# HomeReach Melbourne — 13: Fixed final rental-forecast evaluation

**This notebook scores the reserved final period for the first modelling round.** The method was frozen in commit `d0858bdf46fa6b15c34731f0e41873fb5a2ccfe9`, before this notebook was constructed. The protocol's canonical SHA-256 is `f0168e345aafba1a64b883bef1fe351185d7cf57575ce870f252301b484d1495`.

**Period:** 2023Q4–2025Q3. **Scope:** 31 rental-source LGAs, two-bedroom flats, nominal AUD/week. **Horizon:** one quarter. **Primary comparison:** fixed half-adjusted change Ridge versus last observation. Earlier level Ridge and seasonal naive are secondary references only.

Use the existing **homereach** kernel on **feat/rental-forecast-baselines**. Save the notebook before running from the top. Do not run Code Runner, change the locked protocol, extend previous validation notebooks, or open another writer to these outputs. No new packages or downloads are required.

There are **248 targets per method, 992 predictions and 496 trained fits**. The scaler and trained models refit at each origin with only earlier observations. The first fit uses 94 supervised examples; the last uses 101. Earlier final-period outcomes may enter later origins; **final errors may not change the method**.

This is a retrospective period-based evaluation from a single workbook vintage, not a reconstructed real-time backtest. The outcomes appeared in earlier descriptive EDA: they were reserved from modelling evaluation, not a pristine blind test. There is no significance test, forecast interval, current-rent forecast or automatic application update.

**Outputs:** a new research folder and `docs/rental_final_evaluation.md`. Earlier evidence and the frozen application release are protected. Existing complete final outputs can only be compared and preserved. Conflicting or partial attempts must be investigated and documented, not deleted to obtain a passing run.


## 1. Verify the committed protocol before reading observations

The existing read-only verifier must pass in committed mode. We independently check its content against the pre-scoring commit, verify that commit is an ancestor of the execution HEAD, and record both IDs. This is a configuration/history check—not proof nobody has seen an outcome. The saved notebook source is fingerprinted separately from its cell outputs.

In [ ]:
print('Starting notebook 13 — fixed final-period evaluation...', flush=True)
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import math
import shutil
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import plotly
import plotly.graph_objects as go
from IPython.display import display
from homereach.io.dffh_rent import COLUMN_ORDER, read_panel_csv


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()


def load_json(path):
    value = json.loads(Path(path).read_text(encoding='utf-8-sig'))
    require(isinstance(value, dict), f'Expected a JSON object: {path}')
    return value


def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                    separators=(',', ':'), allow_nan=False).encode()).hexdigest()


ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p/'pyproject.toml').is_file() and (p/'data').is_dir()), None)
require(ROOT is not None, 'Open this notebook inside the existing HomeReach project.')
LOCK_COMMIT = 'd0858bdf46fa6b15c34731f0e41873fb5a2ccfe9'
PROTOCOL_SHA = 'f0168e345aafba1a64b883bef1fe351185d7cf57575ce870f252301b484d1495'
PROTOCOL_PATH = ROOT/'config/rental_final_evaluation_protocol.json'
GATE_PATH = ROOT/'scripts/verify_rental_final_protocol.py'
NOTEBOOK_PATH = ROOT/'notebooks/13_rental_final_evaluation.ipynb'
OUT = ROOT/'data/interim/rental_final_evaluation'
REPORT_PATH = ROOT/'docs/rental_final_evaluation.md'


def git_bytes(*args):
    return subprocess.run(['git', '-C', str(ROOT), *args], capture_output=True,
                          check=True, timeout=30).stdout


# Compare the verifier with its already-recorded version BEFORE importing it.
require(GATE_PATH.is_file() and NOTEBOOK_PATH.is_file(), 'Missing verifier or notebook file; copy the complete update.')
recorded_gate = git_bytes('show', f'{LOCK_COMMIT}:scripts/verify_rental_final_protocol.py')
require(GATE_PATH.read_text(encoding='utf-8-sig').replace('\r\n', '\n') ==
        recorded_gate.decode('utf-8-sig').replace('\r\n', '\n'),
        'The committed protocol verifier has changed. Stop; do not bypass it.')
loader = importlib.util.spec_from_file_location('homereach_final_protocol_gate', GATE_PATH)
gate = importlib.util.module_from_spec(loader)
loader.loader.exec_module(gate)
lock = gate.verify_project(ROOT, require_committed=True)
plan = load_json(PROTOCOL_PATH)
require(canonical_hash(plan) == lock['protocol_sha256'] == PROTOCOL_SHA, 'Frozen protocol identity differs.')
require(lock['protocol_commit'] == LOCK_COMMIT, 'Expected the pre-scoring protocol commit d0858bd; review the history.')
git_bytes('merge-base', '--is-ancestor', LOCK_COMMIT, 'HEAD')
for relative in gate.LOCK_FILES:
    require(gate.same_content(relative, (ROOT/relative).read_bytes(),
                             git_bytes('show', f'{LOCK_COMMIT}:{relative}')),
            f'Frozen file changed since the protocol commit: {relative}')


def saved_notebook_source_hash():
    notebook = load_json(NOTEBOOK_PATH)
    sources = [{'cell_type': c['cell_type'], 'source': ''.join(c['source']).replace('\r\n', '\n')}
               for c in notebook['cells']]
    return canonical_hash({'cells': sources})


CODE_SHA = saved_notebook_source_hash()
EXECUTION_COMMIT = lock['head_commit']
STARTED_AT = datetime.now(timezone.utc).isoformat()
FINAL_QUARTERS = pd.PeriodIndex(plan['periods']['final_target_quarters'], freq='Q-DEC')
VALIDATION_QUARTERS = pd.period_range('2019Q1', '2023Q3', freq='Q-DEC')
MODELS = tuple(m['name'] for m in plan['methods'])
METHOD = {m['name']: m for m in plan['methods']}
LAST = 'last_observation'
CANDIDATE = 'damped_change_ridge_alpha1_weight05'
LEVEL = 'ridge_lags_seasonal_alpha1'
SEASONAL = 'seasonal_naive'
LABELS = {LAST: 'Last observation', CANDIDATE: 'Half-adjusted change Ridge',
          LEVEL: 'Earlier level Ridge', SEASONAL: 'Seasonal naive'}
KEY = ['rental_area_name', 'property_type', 'target_quarter']
SERIES = KEY[:2]
TOLERANCE = plan['primary_comparison']['numeric_tie_tolerance_aud_week']
existing_manifest = None
if OUT.exists():
    require((OUT/'final_manifest.json').is_file(),
            'A prior/partial final-evaluation folder exists without its manifest. Preserve it and investigate; do not delete it.')
    existing_manifest = load_json(OUT/'final_manifest.json')
    require(existing_manifest['protocol_sha256'] == PROTOCOL_SHA and
            existing_manifest['notebook_source_sha256'] == CODE_SHA,
            'An earlier final attempt uses different code or protocol. Preserve it and document any correction; do not overwrite it.')
    print('An earlier final run exists. This execution can verify identical results; it is NOT a fresh unseen test.')
else:
    require(not REPORT_PATH.exists(), 'A final report already exists without its output manifest; preserve and review it.')

print('COMMITTED FINAL PROTOCOL VERIFIED')
print('Protocol commit:', LOCK_COMMIT)
print('Execution HEAD:', EXECUTION_COMMIT)
print('Final targets: 248 per method | Four-method rows: 992 | Planned trained fits: 496')
print('No settings search, deployment, or current-rent forecast will be performed.')


## 2. Verify rental inputs and prior evidence

Read the validated rental CSV and its local quality report. Check all four earlier research manifests and their recorded output files. The model receives only rental history and source references; published counts, transport and geography are not predictor inputs. The full table is read for integrity; every forecast function separately restricts its available history to the forecast origin.

In [ ]:
INPUT = ROOT/plan['source']['processed_input']
QUALITY_PATH = ROOT/plan['source']['local_integrity_report']
SPEC_PATH = ROOT/'config/rent_snapshot.json'
for p in [INPUT, QUALITY_PATH, SPEC_PATH]:
    require(p.is_file(), f'Missing input: {p.relative_to(ROOT)}')
quality = load_json(QUALITY_PATH)
spec = load_json(SPEC_PATH)
SOURCE_SHA = plan['source']['sha256']
require(spec['sha256'] == quality['source']['sha256'] == SOURCE_SHA, 'Rental source vintage differs.')
require(quality['output_sha256'].get(INPUT.name) == sha256_file(INPUT), 'Rental CSV differs from its local pipeline report.')
require(quality.get('csv_readback_matches') is True and quality['quality']['source_values_verified'] == 6572,
        'The earlier rental-input checkpoint is incomplete.')
panel = read_panel_csv(INPUT)
require(panel.columns.tolist() == COLUMN_ORDER and len(panel) == 3286, 'Rental schema or row count changed.')
require(not panel.duplicated(SERIES+['quarter']).any(), 'Duplicate rental key.')
for field, value in {'property_type': '2 bedroom flats', 'area_type': 'LGA', 'source_file': spec['filename'],
                     'source_sheet': spec['sheet'], 'source_reporting_quarter': '2025Q3'}.items():
    require(panel[field].eq(value).all(), f'Unexpected input scope: {field}')
for col in ['median_weekly_rent', 'published_count']:
    require(panel[col].notna().all() and np.isfinite(panel[col].astype(float)).all() and panel[col].gt(0).all(),
            f'Invalid rental input: {col}')
require(panel.source_median_cell.str.fullmatch(r'[A-Z]+[1-9][0-9]*').all(), 'Malformed source-cell references.')
idx = pd.PeriodIndex(panel.quarter, freq='Q-DEC')
np.testing.assert_array_equal(pd.DatetimeIndex(panel.quarter_end).as_unit('ns').asi8,
                             idx.to_timestamp(how='end').normalize().as_unit('ns').asi8)
expected_history = pd.period_range('1999Q2', '2025Q3', freq='Q-DEC').astype(str).tolist()
LGAS = sorted(panel.rental_area_name.unique())
require(len(LGAS) == 31, 'LGA scope changed.')
for _, g in panel.groupby(SERIES, sort=True):
    require(sorted(g.quarter.tolist()) == expected_history, 'Missing or changed quarters in an LGA history.')
# Only these columns can enter the history-only forecast function. Counts and transport are excluded.
HISTORY_COLUMNS = SERIES+['quarter', 'median_weekly_rent', 'source_file', 'source_sheet', 'source_median_cell']
history = panel[HISTORY_COLUMNS].copy()
actual_lookup = panel.set_index(SERIES+['quarter'])

EVIDENCE = {
    'baseline': ('rental_forecast_baselines', 'baseline_manifest.json', 'rental_forecast_baselines.md'),
    'ridge': ('rental_ridge_validation', 'ridge_manifest.json', 'rental_ridge_validation.md'),
    'diagnostics': ('rental_ridge_diagnostics', 'diagnostic_manifest.json', 'rental_ridge_diagnostics.md'),
    'correction': ('rental_correction_validation', 'correction_manifest.json', 'rental_correction_validation.md'),
}
manifests, evidence_paths = {}, []

def checked_child(directory, relative):
    p = directory/relative
    require(not Path(relative).is_absolute() and p.resolve().is_relative_to(directory.resolve()), 'Unsafe manifest path.')
    return p

for label, (folder, manifest_name, report_name) in EVIDENCE.items():
    directory = ROOT/'data/interim'/folder
    manifest_path = directory/manifest_name
    require(manifest_path.is_file(), f'Missing completed earlier experiment: {folder}')
    m = load_json(manifest_path)
    manifests[label] = m
    evidence_paths.append(manifest_path)
    require(m['quality']['csv_readback_matches'] is True, f'Incomplete earlier exports: {folder}')
    for relative, digest in m['output_sha256'].items():
        p = checked_child(directory, relative)
        require(p.is_file() and sha256_file(p) == digest, f'Earlier experiment file changed: {p.relative_to(ROOT)}')
        evidence_paths.append(p)
    report = ROOT/'docs'/report_name
    # Text files can be saved with Windows line endings; preserve their current bytes.
    require(report.is_file(), f'Missing earlier report: {report_name}')
    evidence_paths.append(report)
    if label != 'diagnostics':
        require(m['input_csv_sha256'] == sha256_file(INPUT) and m['source_workbook_sha256'] == SOURCE_SHA,
                f'Earlier experiment used another rental input: {folder}')

b, r, d, c = (manifests[x] for x in ['baseline', 'ridge', 'diagnostics', 'correction'])
base_path = ROOT/'data/interim/rental_forecast_baselines/baseline_manifest.json'
ridge_path = ROOT/'data/interim/rental_ridge_validation/ridge_manifest.json'
diag_path = ROOT/'data/interim/rental_ridge_diagnostics/diagnostic_manifest.json'
require(r['baseline_manifest_sha256'] == d['baseline_manifest_sha256'] == c['baseline_manifest_sha256'] == sha256_file(base_path),
        'Earlier baseline lineage changed.')
require(d['ridge_manifest_sha256'] == c['ridge_manifest_sha256'] == sha256_file(ridge_path), 'Earlier Ridge lineage changed.')
require(c['diagnostic_manifest_sha256'] == sha256_file(diag_path), 'Earlier diagnostic lineage changed.')
pinned = plan['pinned_development_protocols']
require(b['protocol_sha256'] == r['baseline_protocol_sha256'] == pinned['config/rental_forecast_protocol.json']['canonical_json_sha256'],
        'Baseline protocol lineage differs.')
require(r['ridge_protocol_sha256'] == pinned['config/rental_ridge_protocol.json']['canonical_json_sha256'], 'Ridge protocol lineage differs.')
require(d['protocol_sha256'] == pinned['config/rental_ridge_diagnostic_protocol.json']['canonical_json_sha256'], 'Diagnostic protocol lineage differs.')
require(c['protocol_sha256'] == pinned['config/rental_correction_protocol.json']['canonical_json_sha256'], 'Correction protocol lineage differs.')
require(b['quality']['holdout_forecasts'] == b['quality']['holdout_scores'] == 0 and
        r['quality']['final_period_forecasts'] == r['quality']['final_period_scores'] == 0 and
        d['quality']['holdout_forecasts_scored'] == 0 and
        c['quality']['final_period_forecasts'] == c['quality']['final_period_scores'] == 0,
        'Earlier recorded experiments have already scored the final period; review the exposure before proceeding.')

protected = set(evidence_paths + [INPUT, QUALITY_PATH, SPEC_PATH])
for directory in ['config','app','deploy','src','scripts','tests','docs','notebooks']:
    protected.update(p for p in (ROOT/directory).rglob('*') if p.is_file()
                     and '__pycache__' not in p.parts and p.suffix != '.pyc'
                     and p != NOTEBOOK_PATH and p != REPORT_PATH)
raw = ROOT/'data/raw/rent'/spec['filename']
if raw.is_file():
    require(sha256_file(raw) == SOURCE_SHA, 'Original rental workbook changed.')
    protected.add(raw)
parquet = ROOT/'data/processed/rent_quarterly.parquet'
if parquet.is_file(): protected.add(parquet)
input_hashes = {p: sha256_file(p) for p in sorted(protected)}
if existing_manifest:
    require(existing_manifest['input_csv_sha256'] == input_hashes[INPUT], 'Existing final evaluation used a different input.')
    for relative,digest in existing_manifest['output_sha256'].items():
        p=checked_child(OUT,relative)
        require(p.is_file() and sha256_file(p)==digest, f'Existing final evidence changed: {relative}')
    require(REPORT_PATH.is_file() and sha256_file(REPORT_PATH)==existing_manifest['report_sha256'],
            'Existing final report changed; preserve and investigate.')
print('FINAL-EVALUATION INPUTS VERIFIED')
print('Rental rows: 3286 | LGAs: 31 | Quarters: 106')
print('Earlier experiment manifests and output fingerprints: 4/4 verified')
print('Only the rental history is used; no transport or counts enter model inputs.')


## 3. Preserve the validation benchmark

Reconcile the 2,356 saved validation predictions and previous metrics without refitting those models. The earlier baseline and level-Ridge subsets must match the four-method comparison, including actual outcomes and original source-cell references. Final-period errors will be stored separately; do not average them with validation errors to improve a headline.

In [ ]:
def read_evidence(label, filename):
    directory = ROOT/'data/interim'/EVIDENCE[label][0]
    m = manifests[label]
    require(filename in m['output_schemas'], f'Unknown earlier schema: {label}/{filename}')
    schema = m['output_schemas'][filename]
    f = pd.read_csv(directory/filename, dtype=schema, keep_default_na=False)
    require(f.columns.tolist() == list(schema), f'Earlier column order changed: {filename}')
    return f


def metrics(frame, groups):
    rows=[]
    for values,g in frame.groupby(groups,sort=True,dropna=False,observed=True):
        values=(values,) if not isinstance(values,tuple) else values
        errors=g.forecast_weekly_rent_aud.to_numpy(float)-g.actual_weekly_rent_aud.to_numpy(float)
        require(len(errors)>0 and np.isfinite(errors).all(),'Nonfinite or empty forecast errors.')
        rows.append({**dict(zip(groups,values)), 'n_forecasts':len(errors),
                     'mae_aud_week':float(np.abs(errors).mean()),
                     'rmse_aud_week':float(np.sqrt(np.mean(errors**2))),
                     'mean_signed_error_aud_week':float(errors.mean())})
    return pd.DataFrame(rows)


def compare_tables(expected,actual,keys):
    require(expected.columns.tolist()==actual.columns.tolist() and len(expected)==len(actual),'Table shape/schema mismatch.')
    a=expected.sort_values(keys,kind='stable').reset_index(drop=True)
    z=actual.sort_values(keys,kind='stable').reset_index(drop=True)
    for name in a.columns:
        if pd.api.types.is_numeric_dtype(a[name]):
            np.testing.assert_allclose(a[name].astype(float),z[name].astype(float),rtol=1e-10,atol=1e-9,equal_nan=False)
        else:
            require(a[name].astype('string').equals(z[name].astype('string')),f'Text mismatch: {name}')


prior = read_evidence('correction','validation_predictions_all_methods.csv')
old_ridge = read_evidence('ridge','validation_predictions_all_methods.csv')
old_base = read_evidence('baseline','validation_predictions.csv')
CORE=['model',*KEY,'forecast_origin_quarter','forecast_weekly_rent_aud','actual_weekly_rent_aud',
      'target_source_median_cell','error_aud_week','absolute_error_aud_week','squared_error']
require(len(prior)==2356 and not prior.duplicated(['model',*KEY]).any() and set(prior.model)==set(MODELS),
        'Previous four-method comparison coverage changed.')
expected_validation={(area,'2 bedroom flats',str(q)) for area in LGAS for q in VALIDATION_QUARTERS}
for model,g in prior.groupby('model'):
    require(set(map(tuple,g[KEY].to_numpy()))==expected_validation,f'Incomplete previous coverage: {model}')
compare_tables(old_ridge[CORE],prior.loc[prior.model.isin(old_ridge.model.unique()),CORE],['model',*KEY])
compare_tables(old_base[CORE],prior.loc[prior.model.isin([LAST,SEASONAL]),CORE],['model',*KEY])
for label in ['baseline','ridge','correction']:
    method_names={'baseline':[LAST,SEASONAL],'ridge':[LAST,SEASONAL,LEVEL],'correction':list(MODELS)}[label]
    rows=prior.loc[prior.model.isin(method_names)]
    for filename,groups in [('validation_metrics_overall.csv',['model']),
                            ('validation_metrics_by_lga.csv',['model',*SERIES]),
                            ('validation_metrics_by_quarter.csv',['model','target_quarter'])]:
        saved=read_evidence(label,filename)
        recomputed=metrics(rows,groups)
        compare_tables(saved,recomputed[saved.columns],groups)
prior_baseline_checks=0
for row in prior.itertuples(index=False):
    q=pd.Period(row.target_quarter,freq='Q-DEC')
    actual=actual_lookup.loc[(row.rental_area_name,row.property_type,str(q))]
    require(row.forecast_origin_quarter==str(q-1),'Earlier forecast origin differs.')
    require(row.target_source_median_cell==actual.source_median_cell,'Earlier target source-cell mismatch.')
    np.testing.assert_allclose(row.actual_weekly_rent_aud,float(actual.median_weekly_rent),rtol=0,atol=0)
    expected_error=row.forecast_weekly_rent_aud-row.actual_weekly_rent_aud
    np.testing.assert_allclose([row.error_aud_week,row.absolute_error_aud_week,row.squared_error],
                               [expected_error,abs(expected_error),expected_error**2],rtol=1e-10,atol=1e-8)
    if row.model in [LAST,SEASONAL]:
        lag=1 if row.model==LAST else 4
        source=actual_lookup.loc[(row.rental_area_name,row.property_type,str(q-lag))]
        np.testing.assert_allclose(row.forecast_weekly_rent_aud,float(source.median_weekly_rent),rtol=0,atol=0)
        prior_baseline_checks+=1
# Exact cells/quarters in the baseline-specific evidence are also preserved.
for row in old_base.itertuples(index=False):
    q=pd.Period(row.target_quarter,freq='Q-DEC')-(1 if row.model==LAST else 4)
    source=actual_lookup.loc[(row.rental_area_name,row.property_type,str(q))]
    require(row.forecast_source_quarter==str(q) and row.forecast_source_median_cell==source.source_median_cell,
            'Earlier copied-quarter/cell evidence changed.')
print('PRIOR VALIDATION EVIDENCE VERIFIED')
print('Saved comparison rows: 2356 | Historical baseline checks:',prior_baseline_checks)
print('Earlier overall/LGA/quarter metrics: matched | Previous forecasts refitted: 0')


## 4. Define history-only forecasts and independent checks

For a target quarter `t`, select only the same area's observations through `t−1` **before** constructing any features or training labels. The target outcome is not required to issue the prediction. Build four historical rent lags; derive three historical changes for the correction model. Both trained methods standardize within their training data and use Ridge with the settings from the frozen protocol. A separate normal-equation solution checks each trained prediction. Finite negative forecasts are retained and flagged, never silently clipped.

In [ ]:
LEVEL_FEATURES = METHOD[LEVEL]['features']
CHANGE_FEATURES = METHOD[CANDIDATE]['features']
require(LEVEL_FEATURES == [f'rent_lag_{k}' for k in range(1,5)]+[f'target_is_q{k}' for k in [2,3,4]],
        'Unexpected fixed level feature order.')
require(CHANGE_FEATURES == [f'change_lag_{k}' for k in range(1,4)]+[f'target_is_q{k}' for k in [2,3,4]],
        'Unexpected fixed change feature order.')


def prepare_origin(frame, area, property_type, target, minimum_examples=40):
    """Read this series only through t-1. Target/future rent values are never required."""
    q = pd.Period(target, freq='Q-DEC')
    require(set(HISTORY_COLUMNS).issubset(frame.columns), 'Missing history columns.')
    own = frame.loc[frame.rental_area_name.eq(area) & frame.property_type.eq(property_type), HISTORY_COLUMNS].copy()
    own['_q'] = pd.PeriodIndex(own.quarter, freq='Q-DEC')
    past = own.loc[own['_q'].lt(q)].sort_values('_q', kind='stable').reset_index(drop=True)
    require(not past.duplicated(SERIES+['quarter']).any(), 'Duplicate past history key.')
    require(len(past) >= 4+minimum_examples, 'Insufficient earlier supervised history.')
    periods = pd.PeriodIndex(past['_q'], freq='Q-DEC')
    require(periods.equals(pd.period_range(periods[0], q-1, freq='Q-DEC')), 'Gap in past history or missing latest quarter.')
    values = past.median_weekly_rent.to_numpy(dtype=float)
    require(np.isfinite(values).all() and (values > 0).all(), 'Invalid past rent.')
    require(past.source_median_cell.astype('string').str.fullmatch(r'[A-Z]+[1-9][0-9]*').all(), 'Invalid past source cells.')
    require(past.source_file.nunique() == past.source_sheet.nunique() == 1, 'Mixed source provenance within history.')
    train_q = periods[4:]
    lags = np.column_stack([values[4-k:len(values)-k] for k in range(1,5)])
    calendar = np.column_stack([(train_q.quarter==k).astype(float) for k in [2,3,4]])
    final_lags = values[-4:][::-1].copy()
    final_calendar = np.array([float(q.quarter==k) for k in [2,3,4]])
    X_level = np.column_stack([lags,calendar])
    X_change = np.column_stack([lags[:,:3]-lags[:,1:4],calendar])
    predict_level = np.r_[final_lags,final_calendar]
    predict_change = np.r_[final_lags[:3]-final_lags[1:],final_calendar]
    y_level = values[4:].copy()
    y_change = y_level-lags[:,0]
    # Independent dictionary lookups check every historical lag and quarterly change.
    lookup = {str(r.quarter):r for r in past.itertuples(index=False)}
    lag_checks = change_checks = 0
    for i,t in enumerate(train_q):
        expected = np.array([float(lookup[str(t-k)].median_weekly_rent) for k in range(1,5)])
        np.testing.assert_array_equal(lags[i],expected)
        require(t < q, 'A training target reached the forecast target.')
        np.testing.assert_array_equal(X_change[i,:3],expected[:3]-expected[1:])
        lag_checks += 4; change_checks += 3
    expected = np.array([float(lookup[str(q-k)].median_weekly_rent) for k in range(1,5)])
    np.testing.assert_array_equal(final_lags,expected)
    lag_checks += 4; change_checks += 3
    lineage = []
    for k in range(1,5):
        src = lookup[str(q-k)]
        lineage.append({'lag_quarters':k, 'source_quarter':str(q-k), 'rent_aud_week':float(src.median_weekly_rent),
                        'source_file':str(src.source_file),'source_sheet':str(src.source_sheet),
                        'source_median_cell':str(src.source_median_cell)})
    return {'target':q,'periods':periods,'training_quarters':train_q,'last':float(values[-1]),
            'seasonal':float(values[-4]),'X_level':X_level,'X_change':X_change,
            'predict_level':predict_level,'predict_change':predict_change,
            'y_level':y_level,'y_change':y_change,'lineage':lineage,
            'lag_checks':lag_checks,'change_checks':change_checks}


def trained_fit(context, model):
    """Fresh training-only scaling and Ridge fit; independent normal-equation check."""
    require(model in [CANDIDATE,LEVEL], 'Unknown trained method.')
    setting = METHOD[model]
    change_model = model==CANDIDATE
    X = context['X_change' if change_model else 'X_level']
    y = context['y_change' if change_model else 'y_level']
    xp = context['predict_change' if change_model else 'predict_level']
    names = CHANGE_FEATURES if change_model else LEVEL_FEATURES
    require(np.isfinite(X).all() and np.isfinite(y).all() and np.isfinite(xp).all(), 'Nonfinite training value.')
    pipeline = Pipeline([('scale',StandardScaler(with_mean=setting['with_mean'],with_std=setting['with_std'])),
                         ('ridge',Ridge(alpha=setting['alpha'],fit_intercept=setting['fit_intercept'],solver=setting['solver']))])
    pipeline.fit(pd.DataFrame(X,columns=names), y)
    raw_prediction = float(pipeline.predict(pd.DataFrame([xp],columns=names))[0])
    scaler = pipeline.named_steps['scale']; ridge = pipeline.named_steps['ridge']
    n = len(y)
    require(int(scaler.n_samples_seen_)==n, 'Scaler fitted on the wrong number of rows.')
    np.testing.assert_allclose(scaler.mean_,X.mean(axis=0),rtol=1e-12,atol=1e-12)
    np.testing.assert_allclose(scaler.var_,X.var(axis=0),rtol=1e-12,atol=1e-12)
    # Independent scaling is computed directly; constant columns keep scale 1.
    independent_scale = np.sqrt(X.var(axis=0)); independent_scale[independent_scale==0] = 1.0
    np.testing.assert_allclose(scaler.scale_,independent_scale,rtol=1e-12,atol=1e-12)
    z = (X-X.mean(axis=0))/independent_scale
    zbar = z.mean(axis=0); ybar = float(y.mean())
    centered = z-zbar
    coefficients_check = np.linalg.solve(centered.T@centered+setting['alpha']*np.eye(len(names)), centered.T@(y-ybar))
    independent_raw = float(((xp-X.mean(axis=0))/independent_scale-zbar)@coefficients_check+ybar)
    np.testing.assert_allclose(raw_prediction,independent_raw,rtol=1e-9,atol=1e-8)
    forecast = context['last']+setting['damping_weight']*raw_prediction if change_model else raw_prediction
    independent_forecast = context['last']+setting['damping_weight']*independent_raw if change_model else independent_raw
    np.testing.assert_allclose(forecast,independent_forecast,rtol=1e-9,atol=1e-8)
    require(np.isfinite(forecast), 'Nonfinite forecast; stop rather than omit this target.')
    raw_coefficients = ridge.coef_/scaler.scale_
    raw_intercept = float(ridge.intercept_-raw_coefficients@scaler.mean_)
    np.testing.assert_allclose(raw_prediction,raw_intercept+xp@raw_coefficients,rtol=1e-9,atol=1e-8)
    training_q = context['training_quarters']
    audit = {'training_first_target':str(training_q[0]),'training_last_target':str(training_q[-1]),
             'training_rows':n,'scaler_training_rows':int(scaler.n_samples_seen_),
             'earlier_final_targets_in_training':int((training_q>=FINAL_QUARTERS[0]).sum()),
             'training_targets_at_or_after_forecast_target':int((training_q>=context['target']).sum()),
             'alpha':float(setting['alpha']),'raw_model_prediction_aud_week':raw_prediction,
             'independent_raw_prediction_aud_week':independent_raw,'independent_forecast_aud_week':independent_forecast,
             'training_target_mean':ybar,'raw_intercept':raw_intercept,
             'prediction_negative':int(forecast<0)}
    evidence = [{'feature':name,'training_mean':float(scaler.mean_[j]),'training_variance':float(scaler.var_[j]),
                 'training_scale':float(scaler.scale_[j]),'coefficient_standardized':float(ridge.coef_[j]),
                 'coefficient_raw_units':float(raw_coefficients[j]),'forecast_feature_value':float(xp[j])}
                for j,name in enumerate(names)]
    return float(forecast), audit, evidence


def predict_one(frame, area, property_type, target, minimum_examples=40):
    c = prepare_origin(frame,area,property_type,target,minimum_examples)
    key = {'rental_area_name':area,'property_type':property_type,'target_quarter':str(c['target']),
           'forecast_origin_quarter':str(c['target']-1)}
    predictions=[]; fits=[]; coefs=[]
    for model in MODELS:
        if model==LAST: prediction=c['last']
        elif model==SEASONAL: prediction=c['seasonal']
        else:
            prediction,audit,coefficients=trained_fit(c,model)
            fits.append({**key,'model':model,**audit})
            coefs.extend({**key,'model':model,**record} for record in coefficients)
        predictions.append({**key,'model':model,'model_role':METHOD[model]['role'],
                            'last_observed_rent_aud_week':c['last'],
                            'forecast_weekly_rent_aud':float(prediction),'prediction_negative':int(prediction<0)})
    lineage=[{**key,**record} for record in c['lineage']]
    return predictions,fits,coefs,lineage,c['lag_checks'],c['change_checks']


def validate_coverage(frame, target_keys, models):
    require(not frame.duplicated(['model',*KEY]).any(),'Duplicate model/target key.')
    require(set(frame.model)==set(models),'Method coverage differs.')
    for model,group in frame.groupby('model'):
        require(set(map(tuple,group[KEY].to_numpy()))==set(target_keys),'Missing or extra forecast target.')
    require(np.isfinite(frame.forecast_weekly_rent_aud.to_numpy(float)).all(),'Nonfinite prediction.')


def difference_label(delta):
    if delta < -TOLERANCE: return 'candidate_lower_mae'
    if delta > TOLERANCE: return 'last_observation_lower_mae'
    return 'numerically_tied'


def primary_pairs(scored):
    ref=scored.loc[scored.model.eq(LAST),KEY+['actual_weekly_rent_aud','forecast_weekly_rent_aud','absolute_error_aud_week']].copy()
    cand=scored.loc[scored.model.eq(CANDIDATE),KEY+['actual_weekly_rent_aud','forecast_weekly_rent_aud','absolute_error_aud_week']].copy()
    result=ref.merge(cand,on=KEY,how='outer',validate='one_to_one',indicator=True,suffixes=('_last','_candidate'))
    require(result['_merge'].eq('both').all(),'Primary comparison targets differ.')
    np.testing.assert_array_equal(result.actual_weekly_rent_aud_last,result.actual_weekly_rent_aud_candidate)
    result=result.drop(columns=['_merge','actual_weekly_rent_aud_candidate']).rename(columns={'actual_weekly_rent_aud_last':'actual_weekly_rent_aud'})
    result['paired_absolute_error_difference_aud_week']=result.absolute_error_aud_week_candidate-result.absolute_error_aud_week_last
    return result.sort_values(KEY).reset_index(drop=True)


def summarise_pairs(pairs, groups=()):
    rows=[]
    group_iterator=pairs.groupby(list(groups),sort=True,dropna=False,observed=True) if groups else [((),pairs)]
    for values,g in group_iterator:
        values=(values,) if not isinstance(values,tuple) else values
        last=float(g.absolute_error_aud_week_last.mean());candidate=float(g.absolute_error_aud_week_candidate.mean())
        delta=float(g.paired_absolute_error_difference_aud_week.mean())
        np.testing.assert_allclose(delta,candidate-last,rtol=1e-10,atol=1e-10)
        rows.append({**dict(zip(groups,values)),'n_targets':len(g),'last_mae_aud_week':last,
                     'candidate_mae_aud_week':candidate,'candidate_minus_last_mae_aud_week':delta,
                     'relative_mae_improvement_pct':100*(last-candidate)/last if last else np.nan,
                     'sum_paired_loss_difference':float(g.paired_absolute_error_difference_aud_week.sum()),
                     'contribution_to_pooled_gap_aud_week':float(g.paired_absolute_error_difference_aud_week.sum()/len(pairs)),
                     'numeric_comparison':difference_label(delta)})
    return pd.DataFrame(rows)

print('History-only forecast functions defined. Actual target values are attached only after forecasts are produced.')


## 5. Run fictional self-checks

Known examples test lag direction, copied baselines, row ordering, missing quarters, duplicate keys, future-outcome mutation/removal, cross-LGA isolation, constant histories, rolling updates, exact half correction, metric arithmetic and the frozen numerical-tie rule. These are notebook checks, not additional cases in the existing pytest suite.

In [ ]:
self_checks=[]

def check(name,fn):
    fn();self_checks.append(name)


def rejects(fn,text):
    try: fn()
    except ValueError as error:
        require(text in str(error),f'Unexpected error instead of {text}: {error}')
    else: raise ValueError(f'Expected rejection: {text}')


def fictional_series(area='Example A', constant=False):
    qs=pd.period_range('2010Q1',periods=60,freq='Q-DEC')
    v=np.repeat(500.0,len(qs)) if constant else np.array([300+2*i+(i%4)*3 for i in range(len(qs))],float)
    return pd.DataFrame({'rental_area_name':area,'property_type':'2 bedroom flats','quarter':qs.astype(str),
                         'median_weekly_rent':v,'source_file':'fictional.xlsx','source_sheet':'example',
                         'source_median_cell':[f'A{i+1}' for i in range(len(qs))]})

fixture=fictional_series();target=pd.Period('2022Q1',freq='Q-DEC');minimum=8

def predictions_of(frame,t=target):
    return np.array([r['forecast_weekly_rent_aud'] for r in predict_one(frame,'Example A','2 bedroom flats',t,minimum)[0]])

base_predictions=predictions_of(fixture)

def test_known_lags():
    prepared=prepare_origin(fixture,'Example A','2 bedroom flats',target,minimum)
    v=fixture.loc[pd.PeriodIndex(fixture.quarter,freq='Q-DEC')<target,'median_weekly_rent'].to_numpy(float)
    np.testing.assert_array_equal(prepared['predict_level'][:4],v[-4:][::-1])
    np.testing.assert_array_equal(prepared['predict_change'][:3],v[-4:][::-1][:3]-v[-4:][::-1][1:])
    np.testing.assert_array_equal(prepared['predict_level'][4:],[0.,0.,0.])
check('Known lags, changes and Q1 indicators',test_known_lags)

def test_baselines():
    lookup=fixture.set_index('quarter').median_weekly_rent
    ps=predict_one(fixture,'Example A','2 bedroom flats',target,minimum)[0]
    require(ps[0]['forecast_weekly_rent_aud']==lookup[str(target-1)],'Last baseline differs.')
    require(next(p for p in ps if p['model']==SEASONAL)['forecast_weekly_rent_aud']==lookup[str(target-4)],'Seasonal baseline differs.')
check('Both baseline source quarters',test_baselines)

def test_future_mutation():
    changed=fixture.copy();mask=pd.PeriodIndex(changed.quarter,freq='Q-DEC')>=target
    changed.loc[mask,'median_weekly_rent']=1e12
    np.testing.assert_allclose(predictions_of(changed),base_predictions,rtol=0,atol=0)
check('Current/future outcome mutation cannot change issued predictions',test_future_mutation)

def test_removal():
    past=fixture.loc[pd.PeriodIndex(fixture.quarter,freq='Q-DEC')<target]
    np.testing.assert_allclose(predictions_of(past),base_predictions,rtol=0,atol=0)
check('Prediction without target or future outcome rows',test_removal)
check('Row-order invariance',lambda:np.testing.assert_allclose(predictions_of(fixture.sample(frac=1,random_state=7)),base_predictions,rtol=0,atol=0))

def test_unrelated():
    other=fictional_series('Other LGA');other['median_weekly_rent']*=100
    np.testing.assert_allclose(predictions_of(pd.concat([fixture,other],ignore_index=True)),base_predictions,rtol=0,atol=0)
check('No cross-LGA pooling',test_unrelated)
check('Duplicate-history rejection',lambda:rejects(lambda:predictions_of(pd.concat([fixture,fixture.iloc[[5]]],ignore_index=True)),'Duplicate'))
check('Missing-quarter rejection',lambda:rejects(lambda:predictions_of(fixture.drop(index=5)),'Gap'))
check('Missing latest-quarter rejection',lambda:rejects(lambda:predictions_of(fixture.loc[fixture.quarter.ne(str(target-1))]),'Gap'))
check('Insufficient-training rejection',lambda:rejects(lambda:predictions_of(fixture.iloc[40:]),'Insufficient'))

def invalid_rent(value):
    bad=fixture.copy();bad.loc[5,'median_weekly_rent']=value
    rejects(lambda:predictions_of(bad),'Invalid past rent')
check('Nonfinite-history rejection',lambda:invalid_rent(float('nan')))
check('Negative-history rejection',lambda:invalid_rent(-1.0))

def bad_cell():
    bad=fixture.copy();bad.loc[5,'source_median_cell']=''
    rejects(lambda:predictions_of(bad),'Invalid past source')
check('Missing source-cell rejection',bad_cell)
check('Constant-history forecasts stay constant',lambda:np.testing.assert_allclose(predictions_of(fictional_series(constant=True)),[500.]*4,atol=1e-8,rtol=0))

def translation():
    shifted=fixture.copy();shifted.median_weekly_rent+=1000
    np.testing.assert_allclose(predictions_of(shifted),base_predictions+1000,atol=1e-8,rtol=1e-10)
check('Translation invariance of the unchanged rules',translation)

def rolling():
    f=fictional_series();a=prepare_origin(f,'Example A','2 bedroom flats',target,minimum)
    b=prepare_origin(f,'Example A','2 bedroom flats',target+1,minimum)
    require(len(b['y_level'])==len(a['y_level'])+1 and b['training_quarters'][-1]==target,'Incorrect rolling update.')
    qind=f.index[f.quarter.eq(str(target))][0];f.loc[qind,'median_weekly_rent']+=200
    np.testing.assert_allclose(predictions_of(f),base_predictions,rtol=0,atol=0)
    require(predictions_of(f,target+1)[0]!=predictions_of(fixture,target+1)[0], 'Earlier observed outcome not available to next origin.')
check('Earlier outcomes enter only later origins',rolling)

def damping():
    rows,audits,*_=predict_one(fixture,'Example A','2 bedroom flats',target,minimum)
    p=next(r for r in rows if r['model']==CANDIDATE);a=next(r for r in audits if r['model']==CANDIDATE)
    np.testing.assert_allclose(p['forecast_weekly_rent_aud'],p['last_observed_rent_aud_week']+.5*a['raw_model_prediction_aud_week'])
check('Exactly half of the raw predicted change is applied',damping)

def known_metrics():
    f=pd.DataFrame({'model':['x','x'],'forecast_weekly_rent_aud':[2.,6.],'actual_weekly_rent_aud':[1.,3.]})
    r=metrics(f,['model']).iloc[0]
    np.testing.assert_allclose([r.mae_aud_week,r.rmse_aud_week,r.mean_signed_error_aud_week],[2.,math.sqrt(5),2.])
check('Known pooled MAE, RMSE and signed error',known_metrics)

def coverage_tests():
    rows=pd.DataFrame(predict_one(fixture,'Example A','2 bedroom flats',target,minimum)[0])
    keys={('Example A','2 bedroom flats',str(target))}
    validate_coverage(rows,keys,MODELS)
    rejects(lambda:validate_coverage(pd.concat([rows,rows.iloc[[0]]]),keys,MODELS),'Duplicate')
    rejects(lambda:validate_coverage(rows.iloc[:-1],keys,MODELS),'Method')
check('Missing and duplicate forecast coverage rejected',coverage_tests)

def pair_tests():
    f=pd.DataFrame([{'model':m,'rental_area_name':'X','property_type':'2 bedroom flats','target_quarter':'2024Q1',
                     'actual_weekly_rent_aud':10.,'forecast_weekly_rent_aud':v,'absolute_error_aud_week':abs(v-10)}
                   for m,v in [(LAST,12.),(CANDIDATE,11.)]])
    pairs=primary_pairs(f);summary=summarise_pairs(pairs).iloc[0]
    require(summary.candidate_minus_last_mae_aud_week==-1 and summary.relative_mae_improvement_pct==50,
            'Incorrect primary-comparison direction.')
    f.loc[f.model.eq(LAST),'absolute_error_aud_week']=0
    require(pd.isna(summarise_pairs(primary_pairs(f)).iloc[0].relative_mae_improvement_pct),'Zero denominator must be undefined.')
check('Paired-loss direction and zero denominator',pair_tests)

def tie_test():
    require(difference_label(0)=='numerically_tied' and difference_label(TOLERANCE/2)=='numerically_tied'
            and difference_label(-2*TOLERANCE)=='candidate_lower_mae'
            and difference_label(2*TOLERANCE)=='last_observation_lower_mae','Tie policy changed.')
check('Frozen numerical tie tolerance',tie_test)
print(len(self_checks),'final-evaluation self-check groups passed using fictional data.')
print('The actual final-period data has not been scored by these tests.')


## 6. Construct all final-period forecasts

Build the eight-origin plan, then fit both trained methods for each LGA and origin. Progress prints once per LGA. Previous final outcomes can enter only later training windows. All 992 predictions are constructed before their target actuals are joined for scoring. The frozen settings are never chosen by these errors.

In [ ]:
expected_targets={(area,'2 bedroom flats',str(q)) for area in LGAS for q in FINAL_QUARTERS}
require(len(expected_targets)==248 and len(MODELS)==4,'Final target plan differs.')
origin_plan=pd.DataFrame([{'target_quarter':str(q),'forecast_origin_quarter':str(q-1),
                          'training_first_target':'2000Q2', 'training_last_target':str(q-1),
                          'training_rows_per_trained_fit':q.ordinal-pd.Period('2000Q2',freq='Q-DEC').ordinal,
                          'earlier_final_outcomes_available':i,'targets_per_method':31}
                         for i,q in enumerate(FINAL_QUARTERS)])
require(origin_plan.training_rows_per_trained_fit.tolist()==list(range(94,102)),'Planned training sizes differ.')
display(origin_plan)
prediction_records=[];fit_records=[];coefficient_records=[];lineage_records=[]
lag_checks=change_checks=0
for number,area in enumerate(LGAS,start=1):
    for quarter in FINAL_QUARTERS:
        predictions, fits, coefficients, lineage, nlags, nchanges = predict_one(
            history,area,'2 bedroom flats',quarter,plan['rolling_evaluation']['minimum_training_examples'])
        prediction_records.extend(predictions);fit_records.extend(fits)
        coefficient_records.extend(coefficients);lineage_records.extend(lineage)
        lag_checks+=nlags;change_checks+=nchanges
    print(f'{number:02d}/31 {area}: eight targets, four fixed methods completed.',flush=True)
final_predictions=pd.DataFrame(prediction_records).sort_values(['model',*KEY]).reset_index(drop=True)
fit_audit=pd.DataFrame(fit_records).sort_values(['model',*KEY]).reset_index(drop=True)
coefficients=pd.DataFrame(coefficient_records).sort_values(['model',*KEY,'feature']).reset_index(drop=True)
lineage=pd.DataFrame(lineage_records).sort_values([*KEY,'lag_quarters']).reset_index(drop=True)
validate_coverage(final_predictions,expected_targets,MODELS)
require(len(final_predictions)==992 and len(fit_audit)==496 and len(lineage)==992 and len(coefficients)==3224,
        'Final forecast/audit counts differ.')
require(not fit_audit.duplicated(['model',*KEY]).any() and not coefficients.duplicated(['model',*KEY,'feature']).any(),
        'Duplicate fit/coefficient key.')
require(fit_audit.training_rows.min()==94 and fit_audit.training_rows.max()==101,'Training window changed.')
require(fit_audit.training_targets_at_or_after_forecast_target.eq(0).all(),'A fit used current or future outcomes.')
for row in fit_audit.itertuples(index=False):
    q=pd.Period(row.target_quarter,freq='Q-DEC')
    require(row.training_last_target==str(q-1) and row.training_first_target=='2000Q2','Incorrect training boundary.')
    require(row.training_rows==q.ordinal-pd.Period('2000Q2',freq='Q-DEC').ordinal,'Unexpected supervised sample count.')
    require(row.earlier_final_targets_in_training==q.ordinal-FINAL_QUARTERS[0].ordinal,'Final history update rule differs.')
# The current actual rent has not yet been attached to any prediction row.
require('actual_weekly_rent_aud' not in final_predictions,'Actual outcomes entered the prediction table prematurely.')
print('FINAL FORECAST CONSTRUCTION PASSED')
print('Forecast rows: 992 | Trained fits: 496 | Training rows per fit: 94-101')
print('Independent mathematical prediction checks:',len(fit_audit))
print('Final actuals attached for scoring: False')


## 7. Attach final outcomes and reconcile errors

Only now attach the target actuals. Verify source references, target coverage, training limits and independent fitted predictions. MAE is the primary metric. RMSE is computed from all squared errors, not by averaging area RMSEs. Signed error is forecast minus actual. The paired primary difference is candidate absolute error minus last-observation absolute error: **negative favours the candidate**.

In [ ]:
# All forecasts already exist. Attach final outcomes now, by the complete key.
actuals=panel.loc[panel.quarter.isin(FINAL_QUARTERS.astype(str)),
                 SERIES+['quarter','median_weekly_rent','source_file','source_sheet','source_median_cell']].rename(columns={
                     'quarter':'target_quarter','median_weekly_rent':'actual_weekly_rent_aud',
                     'source_median_cell':'target_source_median_cell'})
require(len(actuals)==248 and not actuals.duplicated(KEY).any(),'Final outcomes are incomplete or duplicated.')
scored=final_predictions.merge(actuals,on=KEY,how='left',validate='many_to_one',indicator=True)
require(scored['_merge'].eq('both').all(),'A forecast did not match an actual outcome.')
scored=scored.drop(columns='_merge').sort_values(['model',*KEY]).reset_index(drop=True)
scored['error_aud_week']=scored.forecast_weekly_rent_aud-scored.actual_weekly_rent_aud
scored['absolute_error_aud_week']=scored.error_aud_week.abs()
scored['squared_error']=scored.error_aud_week**2
validate_coverage(scored,expected_targets,MODELS)
# Independent source lineage for all four lags at each final target.
for row in lineage.itertuples(index=False):
    source=actual_lookup.loc[(row.rental_area_name,row.property_type,row.source_quarter)]
    require(pd.Period(row.source_quarter,freq='Q-DEC')==pd.Period(row.target_quarter,freq='Q-DEC')-row.lag_quarters,
            'Forecast source quarter is incorrect.')
    require(source.source_median_cell==row.source_median_cell and source.source_file==row.source_file
            and source.source_sheet==row.source_sheet,'Forecast source provenance differs.')
    np.testing.assert_allclose(float(source.median_weekly_rent),row.rent_aud_week,rtol=0,atol=0)
baseline_source_records=[]
for row in scored.loc[scored.model.isin([LAST,SEASONAL])].itertuples(index=False):
    lag=1 if row.model==LAST else 4
    q=pd.Period(row.target_quarter,freq='Q-DEC')-lag
    source=actual_lookup.loc[(row.rental_area_name,row.property_type,str(q))]
    np.testing.assert_allclose(row.forecast_weekly_rent_aud,float(source.median_weekly_rent),rtol=0,atol=0)
    baseline_source_records.append({'model':row.model,'rental_area_name':row.rental_area_name,
                                   'property_type':row.property_type,'target_quarter':row.target_quarter,
                                   'forecast_origin_quarter':row.forecast_origin_quarter,
                                   'forecast_source_quarter':str(q),'forecast_source_median_cell':source.source_median_cell,
                                   'forecast_source_file':source.source_file,'forecast_source_sheet':source.source_sheet,
                                   'forecast_weekly_rent_aud':row.forecast_weekly_rent_aud})
baseline_sources=pd.DataFrame(baseline_source_records)
require(len(baseline_sources)==496,'Baseline lineage count differs.')
# Check the candidate's applied change and the level prediction against the independent solution.
fit_predictions=scored.loc[scored.model.isin([CANDIDATE,LEVEL]),['model',*KEY,'forecast_weekly_rent_aud']].merge(
    fit_audit[['model',*KEY,'independent_forecast_aud_week']],on=['model',*KEY],validate='one_to_one')
np.testing.assert_allclose(fit_predictions.forecast_weekly_rent_aud,fit_predictions.independent_forecast_aud_week,
                           rtol=1e-9,atol=1e-8)
overall=metrics(scored,['model'])
by_lga=metrics(scored,['model',*SERIES])
by_quarter=metrics(scored,['model','target_quarter'])
require(overall.n_forecasts.eq(248).all() and by_lga.n_forecasts.eq(8).all()
        and by_quarter.n_forecasts.eq(31).all(),'Unequal metric denominators.')
for model in MODELS:
    g=scored.loc[scored.model.eq(model)];m=overall.set_index('model').loc[model]
    np.testing.assert_allclose([m.mae_aud_week,m.rmse_aud_week,m.mean_signed_error_aud_week],
        [math.fsum(g.absolute_error_aud_week)/248,math.sqrt(math.fsum(g.squared_error)/248),math.fsum(g.error_aud_week)/248],
        rtol=1e-12,atol=1e-12)
    for summary in [by_lga,by_quarter]:
        x=summary.loc[summary.model.eq(model)]
        np.testing.assert_allclose(np.average(x.mae_aud_week,weights=x.n_forecasts),m.mae_aud_week,atol=1e-12)
        np.testing.assert_allclose(math.sqrt(np.average(x.rmse_aud_week**2,weights=x.n_forecasts)),m.rmse_aud_week,atol=1e-12)
pairs=primary_pairs(scored)
primary_overall=summarise_pairs(pairs)
primary_lga=summarise_pairs(pairs,SERIES)
primary_quarter=summarise_pairs(pairs,['target_quarter'])
primary=primary_overall.iloc[0]
for grouped in [primary_lga,primary_quarter]:
    require(grouped.n_targets.sum()==248,'Primary group sizes do not cover all targets.')
    np.testing.assert_allclose(grouped.sum_paired_loss_difference.sum(),pairs.paired_absolute_error_difference_aud_week.sum(),atol=1e-9)
    np.testing.assert_allclose(grouped.contribution_to_pooled_gap_aud_week.sum(),primary.candidate_minus_last_mae_aud_week,atol=1e-10)
np.testing.assert_allclose(primary.candidate_minus_last_mae_aud_week,
                           overall.set_index('model').loc[CANDIDATE,'mae_aud_week']-overall.set_index('model').loc[LAST,'mae_aud_week'],atol=1e-10)
print('FINAL SCORES AND PAIRED COMPARISON RECONCILED')
print('Baseline value/quarter/cell checks: 496 | Final lag-source checks: 992')
print('These are final-period results only; validation scores are not pooled with them.')
display(overall.set_index('model').loc[list(MODELS)].reset_index().round(4))
display(primary_overall.round(4))


## 8. Interpret the prespecified primary comparison

Report the size and direction of the pooled difference alongside LGA and quarterly results. A numerical advantage is not a significance claim or a deployment decision. The 1e−9 tolerance only handles arithmetic ties; it is not a practical-importance threshold. The other methods remain secondary references, even when one looks favourable.

In [ ]:
result_text={
 'candidate_lower_mae':'The fixed correction candidate has lower pooled MAE than last observation on the reserved later period.',
 'last_observation_lower_mae':'Last observation has lower pooled MAE than the fixed correction candidate on the reserved later period.',
 'numerically_tied':'The primary methods are numerically tied within the prespecified tolerance; last observation remains the simpler reference.'
}[primary.numeric_comparison]
counts=primary_lga.numeric_comparison.value_counts()
print(result_text)
print(f"Candidate minus last-observation MAE: {primary.candidate_minus_last_mae_aud_week:+.4f} AUD/week")
relative=primary.relative_mae_improvement_pct
print(f"Relative MAE improvement: {relative:.2f}%" if pd.notna(relative) else 'Relative MAE improvement: undefined (reference MAE is zero).')
print('LGA comparisons:',counts.to_dict())
print('Negative predictions retained:',int(scored.prediction_negative.sum()))
print('No significance claim, new parameter search, per-LGA winner selection or automatic deployment decision.')
print('Do not retune against these results and call the rerun a fresh final test.')
display(primary_quarter.round(4))


## 9. Overall and quarterly final-period error

Both charts use the final period only, with identical coverage across methods. The method order follows the protocol rather than implying a recommendation ranking. Every quarterly metric contains the same 31 LGAs.

In [ ]:
figures={}
fig=go.Figure(go.Bar(x=[LABELS[m] for m in MODELS],
                    y=overall.set_index('model').loc[list(MODELS),'mae_aud_week'].to_numpy(),
                    customdata=[[METHOD[m]['role'],248] for m in MODELS],
                    hovertemplate='%{x}<br>Final MAE: $%{y:.4f}/week<br>Role: %{customdata[0]}<br>Targets: %{customdata[1]}<extra></extra>'))
fig.update_layout(title='Final period 2023Q4–2025Q3 · identical 248 targets per method',
                  xaxis_title='Fixed method (not a deployment ranking)',yaxis_title='MAE (AUD/week)',height=480)
figures['01_final_mae.html']=fig
fig.show()
fig=go.Figure()
for model in MODELS:
    g=by_quarter.loc[by_quarter.model.eq(model)].sort_values('target_quarter')
    fig.add_trace(go.Scatter(x=g.target_quarter,y=g.mae_aud_week,name=LABELS[model],mode='lines+markers',
                            hovertemplate='Target quarter: %{x}<br>MAE: $%{y:.4f}/week<extra>%{fullData.name}</extra>'))
fig.update_layout(title='Final-period MAE by target quarter · 31 LGAs in every quarter',
                  xaxis_title='Target quarter',yaxis_title='MAE (AUD/week)',height=600, margin=dict(t=80,b=140),
                  legend=dict(orientation='h',x=0,y=-0.25,yanchor='top'))
figures['02_final_mae_by_quarter.html']=fig
fig.show()


## 10. LGA differences and the fixed Melbourne illustration

The LGA chart compares eight targets for each area. Negative candidate-minus-reference MAE favours the correction. Melbourne is the same example used previously; it is not selected after seeing the final results. Hover identifies the forecast origin and source actual cell.

In [ ]:
g=primary_lga.sort_values(['candidate_minus_last_mae_aud_week','rental_area_name'])
fig=go.Figure(go.Bar(x=g.candidate_minus_last_mae_aud_week,y=g.rental_area_name,orientation='h',
                    customdata=g[['last_mae_aud_week','candidate_mae_aud_week','n_targets']].to_numpy(),
                    hovertemplate='%{y}<br>Candidate minus last MAE: %{x:+.4f}<br>Last: %{customdata[0]:.4f}<br>Candidate: %{customdata[1]:.4f}<br>Targets: %{customdata[2]}<extra></extra>'))
fig.add_vline(x=0)
fig.update_layout(title='Final-period primary comparison by LGA · eight targets each',
                  xaxis_title='Candidate MAE − last-observation MAE (AUD/week); negative favours candidate',
                  yaxis_title='Rental-source LGA',height=1000,margin=dict(l=160))
figures['03_final_candidate_minus_last_by_lga.html']=fig
fig.show()
example=scored.loc[scored.rental_area_name.eq('Melbourne')].sort_values(['model','target_quarter'])
actual_example=example.loc[example.model.eq(LAST)]
fig=go.Figure(go.Scatter(x=actual_example.target_quarter,y=actual_example.actual_weekly_rent_aud,
                        mode='lines+markers',name='Published actual',
                        customdata=actual_example[['target_source_median_cell']].to_numpy(),
                        hovertemplate='Target: %{x}<br>Actual: $%{y:.2f}/week<br>Source cell: %{customdata[0]}<extra></extra>'))
for model in MODELS:
    g=example.loc[example.model.eq(model)]
    fig.add_trace(go.Scatter(x=g.target_quarter,y=g.forecast_weekly_rent_aud,mode='lines+markers',name=LABELS[model],
                            customdata=g[['forecast_origin_quarter','last_observed_rent_aud_week']].to_numpy(),
                            hovertemplate='Target: %{x}<br>Forecast: $%{y:.4f}/week<br>Origin: %{customdata[0]}<br>Last observed: $%{customdata[1]:.2f}<extra>%{fullData.name}</extra>'))
fig.update_layout(title='Melbourne · fixed one-quarter-ahead forecasts in the final period',
                  xaxis_title='Target quarter',yaxis_title='Nominal median weekly rent (AUD/week)',height=620, margin=dict(t=80,b=140),
                  legend=dict(orientation='h',x=0,y=-0.25,yanchor='top'))
figures['04_melbourne_final_forecasts.html']=fig
fig.show()
print('Four final-period charts created. Melbourne is the same illustration used in earlier notebooks, not a selected winner.')


## 11. Export without replacing earlier evidence

Save 13 CSV evidence tables, four interactive HTML charts, the protocol snapshot, manifest and report. Every CSV is reloaded and compared. Earlier inputs and application files are fingerprint-checked again. A complete identical rerun compares and preserves the first report and timestamps. A changed code/protocol or incomplete prior run stops for review. Staged export files are retained on failure so an original attempt is not quietly erased. Publishing the results and docs is not a multi-file database transaction; do not run simultaneous writers.

In [ ]:
exports={
    'final_predictions.csv':scored,
    'final_metrics_overall.csv':overall,
    'final_metrics_by_lga.csv':by_lga,
    'final_metrics_by_quarter.csv':by_quarter,
    'primary_paired_errors.csv':pairs,
    'primary_comparison_overall.csv':primary_overall,
    'primary_comparison_by_lga.csv':primary_lga,
    'primary_comparison_by_quarter.csv':primary_quarter,
    'fit_audit.csv':fit_audit,
    'preprocessing_and_coefficients.csv':coefficients,
    'forecast_feature_lineage.csv':lineage,
    'baseline_source_checks.csv':baseline_sources,
    'origin_plan.csv':origin_plan,
}


def frame_schema(frame):
    return {name:('int64' if pd.api.types.is_integer_dtype(dtype) else
                  'float64' if pd.api.types.is_float_dtype(dtype) else 'string')
            for name,dtype in frame.dtypes.items()}


def read_export(path,schema):
    nulls={name:[''] for name,dtype in schema.items() if dtype=='float64'}
    return pd.read_csv(path,dtype=schema,keep_default_na=False,na_values=nulls)


def check_roundtrip(expected,actual):
    require(expected.columns.tolist()==actual.columns.tolist() and len(expected)==len(actual),'CSV shape changed.')
    for name,kind in frame_schema(expected).items():
        if kind=='float64':
            np.testing.assert_allclose(expected[name].to_numpy(float),actual[name].to_numpy(float),
                                       rtol=1e-10,atol=1e-9,equal_nan=True)
        elif kind=='int64':np.testing.assert_array_equal(expected[name].to_numpy(),actual[name].to_numpy())
        else:require(expected[name].astype('string').reset_index(drop=True).equals(actual[name].astype('string').reset_index(drop=True)),
                     f'CSV text differs: {name}')


def verify_protected():
    for path,digest in input_hashes.items():
        require(path.is_file() and sha256_file(path)==digest,f'Protected input changed: {path.relative_to(ROOT)}')
    require(saved_notebook_source_hash()==CODE_SHA,'Notebook source changed during execution; preserve the attempt and investigate.')
    require(git_bytes('rev-parse','HEAD').decode().strip()==EXECUTION_COMMIT,'Git HEAD changed during the evaluation.')
    gate.verify_project(ROOT,require_committed=True)


verify_protected()
versions={'python':sys.version.split()[0],'pandas':pd.__version__,'numpy':np.__version__,
          'scikit_learn':sklearn.__version__,'plotly':plotly.__version__}
quality_final={'lgas':31,'final_quarters':8,'targets_per_method':248,'comparison_rows':992,
               'trained_fits':len(fit_audit),'independent_prediction_checks':len(fit_audit),
               'historical_lag_checks':lag_checks,'historical_change_checks':change_checks,
               'forecast_lag_source_checks':len(lineage),'baseline_source_checks':len(baseline_sources),
               'earlier_validation_rows_verified':len(prior),'prior_validation_baseline_checks':prior_baseline_checks,
               'self_check_groups':len(self_checks),'csv_tables':len(exports),'charts':len(figures),
               'csv_readback_matches':True,'identical_target_coverage':True,'paired_loss_reconciliation':True,
               'final_period_forecasts_scored':len(scored),'negative_predictions_retained':int(scored.prediction_negative.sum()),
               'parameter_combinations_searched':0,'targets_removed':0,'transport_features_used':False,
               'current_rent_forecast_produced':False,'model_pickle_written':False,'protected_files_unchanged':True}
report_lines=['# HomeReach Melbourne — Fixed final rental-forecast evaluation','',
              f'Execution started (UTC): {STARTED_AT}','',
              '## Pre-scoring protocol and scope','',
              f'- Frozen protocol commit: `{LOCK_COMMIT}`.',f'- Execution HEAD: `{EXECUTION_COMMIT}`.',
              f'- Protocol SHA-256 (canonical JSON): `{PROTOCOL_SHA}`.',f'- Saved notebook source SHA-256: `{CODE_SHA}`.',
              '- Final targets: 2023Q4–2025Q3; 31 source-group LGAs; two-bedroom flats; nominal AUD/week.',
              '- Primary comparison: the fixed damped-change candidate versus last observation. Other methods are secondary references, not post-hoc replacements.',
              '- All four methods have exactly 248 forecasts. Validation and final scores are kept separate.','',
              '## Final-period metrics','',
              '| Method | Role | Targets | MAE (AUD/week) | RMSE (AUD/week) | Mean signed error (forecast - actual) |',
              '|---|---|---:|---:|---:|---:|']
for model in MODELS:
    row=overall.set_index('model').loc[model]
    report_lines.append(f'| {LABELS[model]} | {METHOD[model]["role"]} | {int(row.n_forecasts)} | {row.mae_aud_week:.4f} | {row.rmse_aud_week:.4f} | {row.mean_signed_error_aud_week:.4f} |')
report_lines += ['', '## Primary paired comparison','',result_text,
    f'- Candidate minus last-observation MAE: {primary.candidate_minus_last_mae_aud_week:+.6f} AUD/week (negative favours candidate).',
    f'- Relative MAE improvement: {relative:.4f}%.' if pd.notna(relative) else '- Relative MAE improvement: undefined because reference MAE is zero.',
    f'- Lower candidate MAE in {int(counts.get("candidate_lower_mae",0))} LGAs; lower last-observation MAE in {int(counts.get("last_observation_lower_mae",0))}; numerical ties in {int(counts.get("numerically_tied",0))}.',
    f'- Numerical tie tolerance: {TOLERANCE:g} AUD/week. This is not a practical-importance threshold.',
    '- Descriptive error differences only. No significance test, iid confidence interval, prediction interval or automatic deployment rule.',
    '- The final outcomes must not be used to retune features, alpha, damping, training window or per-LGA model choices.','',
    '## Fixed rolling procedure','',
    'For target t, every method uses only this LGA/property history dated at or before t-1. Predictions are formed before target actuals are attached.',
    'The two trained methods refit StandardScaler and Ridge at every origin on 94–101 earlier supervised examples. First training target: 2000Q2. Alpha=1, intercept enabled, SVD solver.',
    'Level Ridge uses four rent lags and target-quarter indicators. The correction model uses three lagged changes and target-quarter indicators; forecast = last rent + 0.5 × predicted change.',
    'Earlier final-period outcomes enter only later origins, following the committed one-step procedure. No model setting is selected using final errors.',
    'Last observation copies t-1; seasonal naive copies t-4. Both are one-quarter-ahead forecasts. No clipping, rounding, deadband or outcome-direction oracle.',
    f'Finite negative predictions retained and flagged: {int(scored.prediction_negative.sum())}. Nonfinite predictions or missing targets stop the evaluation.','',
    '## Verification','',
    f'- {len(prior):,} prior validation comparison rows verified; earlier metrics and files preserved.',
    f'- {prior_baseline_checks:,} prior baseline predictions independently matched to historical values.',
    f'- {lag_checks:,} historical lag comparisons and {change_checks:,} historical-change checks across final origins.',
    '- 496 baseline copied-value/quarter/cell checks; 992 final-target lag/source-cell records.',
    '- 496 training-only scaler and independent mathematical Ridge prediction checks; 3,224 feature/coefficient records.',
    '- Exact target coverage, per-LGA/per-quarter metric denominators, pooled RMSE and paired-loss contributions reconciled.',
    f'- {len(self_checks)} fictional self-check groups; these are notebook checks, not added pytest cases.',
    f'- {len(exports)} CSV tables reloaded and matched; {len(figures)} final-period charts.',
    '- Original rental files, previous experiments, application, registry and packaged release unchanged.','',
    '## Interpretation boundaries','',*['- '+text for text in plan['limitations']],
    '- The candidate was designed after validation diagnostics. Final-period errors are a limited later-period assessment, not a new tuning opportunity.',
    '- MAE is error in an area-level weekly-rent statistic. It is not a weekly forecast of individual property prices or a live/current-rent forecast.',
    '- A favourable numeric result alone does not justify online model promotion. The deployed explorer remains descriptive.',
    '- Identical reruns only verify saved evidence. A bug fix must preserve and document the original attempt and outcome exposure; it is not a fresh unseen test.','',
    '## Reproduction','',
    'Run notebooks/13_rental_final_evaluation.ipynb on feat/rental-forecast-baselines after the committed protocol check.',
    'Outputs: data/interim/rental_final_evaluation/. Existing complete results are compared and preserved, not silently overwritten.',
    'If a previous/partial run conflicts with the code or input lineage, stop, preserve it and document the issue before another evaluation.',
    f'Runtime: {json.dumps(versions,sort_keys=True)}','',
    '## Primary references','',*['- '+url for url in plan['references']],
    '- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html',
    '- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html','']
REPORT_TEXT='\n'.join(report_lines)
output_schemas={name:frame_schema(frame) for name,frame in exports.items()}
if existing_manifest:
    require(existing_manifest['earlier_manifest_sha256']=={
        label:sha256_file(ROOT/'data/interim'/info[0]/info[1]) for label,info in EVIDENCE.items()},
        'Previous final evidence came from different development manifests; preserve it.')
    for filename,frame in exports.items():
        require(existing_manifest['output_schemas'][filename]==output_schemas[filename],'Previous final schema differs.')
        check_roundtrip(frame,read_export(OUT/filename,output_schemas[filename]))
    FILES_WRITTEN=0
    print('REPEAT VERIFICATION PASSED: all 13 tables match; original outputs, report and timestamp preserved.')
else:
    OUT.parent.mkdir(parents=True,exist_ok=True)
    stage=Path(tempfile.mkdtemp(prefix='.rental-final-stage-',dir=OUT.parent))
    try:
        for filename,frame in exports.items():
            frame.to_csv(stage/filename,index=False,encoding='utf-8',lineterminator='\n')
            check_roundtrip(frame,read_export(stage/filename,output_schemas[filename]))
        (stage/'figures').mkdir()
        for filename,figure in figures.items():
            figure.write_html(stage/'figures'/filename,include_plotlyjs=True,full_html=True,
                              config={'responsive':True,'displaylogo':False})
        (stage/'final_protocol.json').write_text(json.dumps(plan,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
        (stage/'final_evaluation_report.md').write_text(REPORT_TEXT,encoding='utf-8')
        verify_protected()
        manifest={'created_at_utc':STARTED_AT,'builder':'notebooks/13_rental_final_evaluation.ipynb',
                  'protocol_commit':LOCK_COMMIT,'execution_commit':EXECUTION_COMMIT,
                  'protocol_sha256':PROTOCOL_SHA,'notebook_source_sha256':CODE_SHA,
                  'research_branch':lock['branch'],'source_workbook_sha256':SOURCE_SHA,
                  'input_csv_sha256':input_hashes[INPUT],
                  'earlier_manifest_sha256':{label:sha256_file(ROOT/'data/interim'/info[0]/info[1]) for label,info in EVIDENCE.items()},
                  'input_sha256':{p.relative_to(ROOT).as_posix():h for p,h in input_hashes.items()},
                  'versions':versions,'quality':quality_final,'self_checks':self_checks,
                  'final_metrics':overall.to_dict(orient='records'),
                  'primary_comparison':primary_overall.astype(object).where(pd.notna(primary_overall),None).to_dict(orient='records'),
                  'output_schemas':output_schemas,
                  'output_sha256':{p.relative_to(stage).as_posix():sha256_file(p) for p in sorted(stage.rglob('*')) if p.is_file()},
                  'report_sha256':sha256_file(stage/'final_evaluation_report.md'),
                  'interpretation':'Fixed later-period descriptive evaluation; no automatic promotion or new tuning.'}
        (stage/'final_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf-8')
        require(not OUT.exists() and not REPORT_PATH.exists(),'Output destination appeared during the run; stop simultaneous writers.')
        stage.rename(OUT)
        REPORT_PATH.parent.mkdir(parents=True,exist_ok=True)
        with REPORT_PATH.open('x',encoding='utf-8',newline='\n') as handle:handle.write(REPORT_TEXT)
        FILES_WRITTEN=len(list(p for p in OUT.rglob('*') if p.is_file()))+1
        print('Thirteen CSV round trips passed; four charts, protocol snapshot, report and manifest saved.')
    except Exception:
        print('Export did not complete. Preserve the staged/final evidence for investigation:',stage,OUT,flush=True)
        raise
verify_protected()
print('Previous experiments and protected inputs remained unchanged.')


## 12. Final checkpoint

Save the notebook with Ctrl+S after completion. Send this checkpoint and the four-method table. Do not change the method in response to the result or merge forecasts into main. A later redesign requires new evidence, and any implementation correction must disclose the original outcome exposure.

In [ ]:
print('FINAL RENTAL EVALUATION CHECKPOINT PASSED')
print('Protocol lock commit:',LOCK_COMMIT[:7],'| Execution HEAD:',EXECUTION_COMMIT[:7])
print('Final period: 2023Q4-2025Q3 | LGAs: 31 | Quarters: 8')
print('Forecasts per method: 248 | Four-method comparison rows: 992')
print('Trained LGA/origin models: 496 | Earlier training rows per fit: 94-101')
print('Independent historical lag checks:',lag_checks,'| Lag-change checks:',change_checks)
print('Independent mathematical prediction checks:',len(fit_audit))
print('Baseline value/quarter/cell checks:',len(baseline_sources),'| Final lag-source records:',len(lineage))
print('Earlier validation comparison rows preserved:',len(prior),'| Previous metrics: matched')
print('Identical target coverage and primary paired-loss contributions: reconciled')
print('Final-evaluation self-check groups:',len(self_checks),'passed')
print('CSV tables saved and reloaded:',len(exports),'| Interactive charts:',len(figures))
print('Final-period forecasts scored:',len(scored),'| Validation targets added to final metrics: 0')
print('Parameter combinations searched: 0 | Targets removed: 0')
print('Transport features used: False | Current-rent forecast produced: False')
print('Protected inputs, prior experiments and deployed release unchanged: True')
print('Saved report:',REPORT_PATH.relative_to(ROOT))
print('Saved outputs:',OUT.relative_to(ROOT))
print('Identical prior outputs preserved:',existing_manifest is not None)
display(overall.set_index('model').loc[list(MODELS)].reset_index().round(4))
display(primary_overall.round(4))
print(result_text)
print('This does not automatically promote a forecast model to the live application.')


## Primary references

The method is fixed by `config/rental_final_evaluation_protocol.json`; these references explain the underlying tools and evaluation concepts.

- [Rolling-origin evaluation](https://otexts.com/fpp3/tscv.html)
- [Avoiding information leakage](https://scikit-learn.org/stable/common_pitfalls.html)
- [Ridge regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)
- [Training-only StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

A favourable number is descriptive evidence for these eight quarters. It is not a model-selection licence, an estimate of today's rent, or a guarantee for future dates.